# Binary Planet Suppression: Imports and data gathering

In [1]:
from astropy.table import Table
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

mpl.rc('lines',linewidth = 1.5)
mpl.rc('font',size = 12)
mpl.rc('axes',labelsize = 14, linewidth=1.25)
mpl.rc('xtick',labelsize = 14)
mpl.rc('ytick',labelsize = 14)
# enable math fonts
mpl.rc('mathtext', default = 'regular')

import pickle

import pandas as pd

## Data from Kepler survey

Our goal here is just to get parallaxes for the KOIs. I only need these for the Sullivan binary catalog later, because for those stars I will need to convert angular separations to distances.

### First up the KOI catalog, downloaded from NExSci

In [2]:
# read in KOI catalog, massage the name strings
kois = Table.read('/Users/enewton/DATA_DIR/nexscikois_2025.05.12_09.55.50.vot').to_pandas()
kois['kepid'] = kois['kepid'].astype('int64')
kois['KOI'] = kois['kepoi_name'].str.extract(r'K0*(\d+)\.\d+')[0].astype(int)

print("Length of KOI table: ", len(kois))
kois[['kepid','KOI', 'kepoi_name', 'kepler_name', 'koi_disposition', 'koi_prad']].head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/enewton/DATA_DIR/nexscikois_2025.05.12_09.55.50.vot'

In [ ]:
kois.columns

### Now Bedell's Gaia-Kepler fun crossmatches

First we'll look at the 1-1 matches, `kepler_dr3_good.fits`. 

In [ ]:
# Read in the KIC from Bedell's Gaia-Kepler fun (1-1 matches only, all entries are unique)
kic = Table.read('/Users/enewton/DATA_DIR/kepler_dr3_good.fits').to_pandas()
kic['bedell_cat'] = 'good'
print("Length of KICxGaia table: ", len(kic))

# Cross match KOIs and KIC 
kois_x_kic = kois.merge(kic[['kepid', 'parallax','parallax_over_error','source_id','kepler_gaia_ang_dist',
                             'phot_g_mean_mag','ruwe','teff','bedell_cat']], how='left', on='kepid')
print("Length of KOIsxKICxGaia table: ", len(kois_x_kic))

# note: converts source_id to float because of NaNs

In [ ]:
kic.columns

Now we'll move onto the larger cross-match Megan did, based on larger angular distances. I determined I needed the 4 arcsec table (too many stars were outside the 1 arcsec match). My by-eye checking of several entries on Simbad indicates that the additionally matches are at least in agreement with what Simbad thinks. They're all pretty faint, seem to be distant M dwarfs. Would be ideal to check the colors actually make sense, but that's beyond what's needed for this analysis.

In [ ]:
## Read in the bigger cross-match to find the missing Gaia matches
kic4as = Table.read('/Users/enewton/DATA_DIR/kepler_dr3_4arcsec.fits').to_pandas()
kic4as['bedell_cat'] = '4as'

# Fill in the stars with missing Gaia matches
missing_mask = kois_x_kic['source_id'].isnull()
missing_rows = kois_x_kic.loc[missing_mask, 'kepid'].unique()

# Subset of kic1as those missing parallaxes
kic4as_subset = kic4as[kic4as['kepid'].isin(missing_rows)]

# Hope the closest match is the best match
kic4as_first = kic4as_subset.sort_values(['kepid','kepler_gaia_ang_dist']).groupby('kepid').first().reset_index()

# Columns coming from kic1as_first to fill into kic_merged
columns_to_fill = ['parallax', 'parallax_over_error','source_id', 'kepler_gaia_ang_dist', 'phot_g_mean_mag','ruwe','teff','bedell_cat']

# Merge relevant kic1as columns with kic_updated
k1 = kic4as_first[['kepid'] + columns_to_fill]
temp = kois_x_kic.merge(k1, how='left', on='kepid', suffixes=('', '_4as'))

for col in columns_to_fill:
    temp[col] = temp[col].fillna(temp[col + '_4as']) # fill column
    temp.drop(columns=[col + '_4as'], inplace=True) # drop temp column

# Temp now has the filled columns
kois_x_kic4as = temp
    
kois_x_kic4as.loc[kois_x_kic4as['parallax'] < 0, 'parallax'] = np.nan

kois_x_kic4as[columns_to_fill][missing_mask].head() #2126519126656859904

print("Length of updated KOIsxKICxGaia table: ", len(kois_x_kic4as))
print("Previously had missing matches: ", sum(missing_mask))
print("Now have missing matches: ", sum(kois_x_kic4as['source_id'].isnull()))
print("Now have missing number of parallaxes: ", sum(kois_x_kic4as['parallax'].isnull()))

### Save the results

In [ ]:
kois_x_kic4as[['kepid', 'kepoi_name', 'kepler_name', 'koi_disposition',
               'koi_pdisposition', 'koi_period', 'koi_prad', 'koi_prad_err1', 'koi_prad_err2',
               'koi_kepmag', 'KOI', 'koi_steff', 'teff',
               'source_id', 'parallax', 'parallax_over_error', 'kepler_gaia_ang_dist',
               'phot_g_mean_mag', 'ruwe','bedell_cat']].to_pickle('kois+gaia.pkl')


## Data from Sullivan et al.

### Starting with the published tables

In [ ]:
#https://vizier.cds.unistra.fr/viz-bin/VizieR-3?-source=J/AJ/168/129/table5
sullivan = Table.read('sullivan2024_planets.vot').to_pandas()

# https://vizier.cds.unistra.fr/viz-bin/VizieR-3?-source=J/AJ/168/129/sample
stars = Table.read('sullivan2024_stars.vot').to_pandas() 

planets = sullivan.merge(stars, how='left', on='KOI')
planets['KOI'] = planets['KOI'].astype(int)

In [ ]:
print("Length of sullivan: ", len(sullivan), " (This is the number of planets in the binary sample)")
sullivan.head()

In [ ]:
print("Length of stars: ", len(stars), " (This is the number of stars/planetary SYSTEMS in the binary sample)")
print("Confirm total number of planets: ", sum(stars['Np']), " (This is the sum of the number of planets column)")
stars.head()

In [ ]:
# massage the KOI names

koi = planets['KOIpl'].astype(float)

scaled = (koi * 100).round().astype(int)            # e.g. 4201 for 42.01
whole = (scaled // 100).astype(str).str.zfill(5)    # zero-pad whole part to 5 digits
frac2 = (scaled % 100).astype(str).str.zfill(2)     # two-digit fractional part

planets['kepoi_name'] = 'K' + whole + '.' + frac2

In [ ]:
print("Length of planets after merging sullivan+stars: ", len(planets))
planets[['KOIpl','KOI','kepoi_name']].head()

### Getting distances by any means necessary for Sullivan sample

Most of these are wide enough that Gaia has totally fine parallaxes. Some are too close. Kendall got distance estimates for these (not published, but given to me in the derived_star table) that I'll use when I have to.

Start with just cross-matching to the KOIs+Gaia list I just created.

In [ ]:
# read in the file I created above
with open('kois+gaia.pkl', 'rb') as file:
       kgx = pickle.load(file)
    
# merging Kendall's catalog with KOIs+gaia match
s24_x_gaia = planets.merge(kgx, how='left', on=['kepoi_name','KOI'])

# doing distance calculation
s24_x_gaia['distance'] = 1000./s24_x_gaia['parallax']

print("In the Sullivan sample, we have missing number of parallaxes: ", sum(s24_x_gaia['parallax'].isnull()))

The rest should have distances here

In [ ]:
extra = pd.read_csv(open('derived_star.tex', 'r'), sep='&')
extra['KOI'] = extra['sname'].astype('int64')

#s24_x_gaia.merge(extra, how='left', )

In [ ]:
binary_db = s24_x_gaia.merge(extra, how='left', on='KOI', suffixes=['','_extra'])


In [ ]:
# borrowed from https://www.eso.org/~hboffin/Teaching/gaia_plx-2.html

def plotdis(rel_err):
    plx = 1. # in mas; this value doesn't really matter, so we can keep as such
    print(f'Parallax= {plx}, with a relative error of {rel_err}%')
    
    error_plx = rel_err / 100. * plx
    test_par= np.random.normal(plx,error_plx,100000)   # Compute a distribution of parallaxes
    test_dis = 1000./test_par                          # Distances is 1000 / parallax in pc
    test_dis = test_dis [test_dis > 0.]                # Remove the non-physical negative distances

    #print the miminum and maximum distances 
    print ("Distances vary between ",np.round(test_dis.min(),1),"and",np.round(test_dis.max(),1)," pc")

    # Compare to a Gaussian distribution of distances, centred on 1/plx 
    # and with an error given by  
    # error_dis = error_plx / plx**2. 
    dis = 1./plx
    error_dis = error_plx *dis**2. 
    
    # this is in kpc, so multiply by 1000.
    dis, error_dis = 1000.*dis, 1000.*error_dis
    
    xdis = np.linspace(-3.*error_dis,3.*error_dis,100)
    ydis= np.exp(-0.5*(xdis/error_dis)**2)
    
    
    #Compute the histogramme of distances
    bin0 = np.percentile(test_dis,.1)
    bin1 = np.percentile(test_dis,99.9)
    if rel_err > 20:
        bin0 = 0.
        bin1 = 3000./plx
    
    z, bin = np.histogram(test_dis,bins=100,range=(bin0,bin1)) #dis-3.*error_dis,dis+5.*error_dis))
    mo = bin[np.argmax(z)]   # compute the maximum of the histogram, i.e. the mode
    plt.vlines(mo,0,z.max(),color='yellow',label='Mode') 
    med = np.percentile(test_dis,50)   # the median value
    plt.vlines(med,0,z.max(),color='magenta',label='Median')
    
    ydis = ydis * z.max()
    plt.plot(xdis+dis,ydis, label='Formal Gaussian')
    
    plt.hist(test_dis,bins=bin,label='Distribution')
    print("Formal distance: ",np.round(dis,0),"+/-",error_dis," pc")
    print("Mean distance: ",np.round(np.mean(test_dis),0),"+/-",np.round(test_dis.std(),0)," pc")
    print ("Median of distance distribution: ",np.round(med,0)," pc")
    print ("Mode of distance distribution: ",np.round(mo,0)," pc")
    
    plt.xlim(bin0,bin1)
    plt.title(f'Distribution of distances with plx={plx} and error={rel_err}%')
    plt.legend();
    #print('------------------------------------------')

relative_error_plx = 1./20. * 100.  # in percents
plotdis(relative_error_plx)

In [ ]:
plt.scatter(binary_db['distance'], binary_db['distance_extra'], c='gray', label='All stars')
plt.scatter(binary_db['distance'][binary_db['parallax_over_error']<20], 
            binary_db['distance_extra'][binary_db['parallax_over_error']<20],
           c='r', label=">5% error")
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Distance from 1/parallax (pc)')
plt.ylabel('Sullivan distance (pc)')
plt.legend()
plt.colorbar()

In [ ]:
# replace where parallax error > 5% or not available at all!
replace = (binary_db['parallax_over_error'] < 20) | (~np.isfinite(binary_db['distance']))
binary_db.loc[replace, 'distance'] = binary_db['distance_extra'][replace]

binary_db['SepAU'] = binary_db['distance']*binary_db['Sep']

binary_db.to_pickle('kendall+kepler.pkl')

# Assessing the compiled catalogs

Making sure things look right

### Make my catalogs

In [4]:
# read in Kendall's catalog, xmatched to all the stuff from Kepler that I need
with open('kendall+kepler.pkl', 'rb') as file:
       db = pickle.load(file)

# read in my kois table
with open('kois+gaia.pkl', 'rb') as file:
        kois = pd.read_pickle(file)


FileNotFoundError: [Errno 2] No such file or directory: 'kendall+kepler.pkl'

### Radius distribution of all KOIs (less the binaries)

Need the filtering on errors in order to see the radius gap.

In [3]:
# trying to confirm this matches Sullivan's
# seems good, prob a bit dif b/c using different planet param database


good = (db['e_Rpkep']/db['Rpkep'] < 0.15) & (db['koi_period']<100)
close = db['SepAU']<100

plt.scatter(db['SepAU'][good], db['Rpkep'][good])
plt.scatter(db['SepAU'][good & close], db['Rpkep'][good & close])
plt.ylim(0.4,4.1)
plt.xlim(3, 4000)
plt.xscale('log')
plt.xlabel('Binary sep (AU)')
plt.ylabel('Rp (Earth rad)')

NameError: name 'db' is not defined

In [ ]:
#kendall assumes they orbit primary stars which is prob mostly true based on nathanael's work

# planets with good radii in binary systems with rho<100 au and planet period < 100 days
plt.figure(figsize=(6,2))
plt.hist(db['Rppri'][good & close], range=[0.2,4], bins=20, 
         density=True, histtype='step',
        label='Close binaries')

# compared to the kois that are at least not definite binaries with good radii and per<100 days
plt.hist(kois_less['koi_prad'][kgood], range=[0.2,4], bins=20, 
         density=True, histtype='step',
        label='Single stars')

plt.legend()
plt.xlabel('Rp')
plt.ylabel('Density')
plt.show()

plt.figure(figsize=(6,2))
plt.hist(db['Rppri'][good & ~close], range=[0.2,4], bins=20, 
         density=True, histtype='step',
        label='Wide binaries')
plt.hist(kois_less['koi_prad'][kgood], range=[0.2,4], bins=20, 
         density=True, histtype='step',
        label='Single stars')

plt.legend()
plt.xlabel('Rp')
plt.ylabel('Density')
plt.show()
